# Bayesian Optimization for Black-Box Functions (Submission 4)

This notebook implements the advanced Bayesian Optimization framework for **Round 4 Queries** across 8 black-box functions. 

### Rationale & Incremental Updates for Round 4:
* **Matern 5/2 Kernel:** Maintained for structural flexibility over complex landscape irregularities.
* **Dynamic Exploration Jitter ($\xi$ Strategy):** Introduced to adaptively shift behavior based on intermediate state convergence:
  * **Low $\xi$ (0.001) for SUCCESS:** Exploiting prominent peaks uncovered during Round 3 (Functions 5, 7, 8).
  * **High $\xi$ (0.100) for STUCK:** Forcing broad exploration where metrics yielded negative boundaries or flat stagnation (Functions 1, 3, 4, 6).
  * **Medium $\xi$ (0.010) for STEADY:** Regular progressive stepping (Function 2).
* **Expanded Acquisition Search Surface:** Configured to execute **50 random restarts** of the local L-BFGS-B optimization engine to meticulously probe the parameter spaces as the collected history reaches deeper maturity.

In [ ]:
import numpy as np
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm

# Ensure inline plotting
%matplotlib inline 

print("Environment ready. Core computation libraries loaded.")

## The Bounded Bayesian Optimizer Architecture

In [ ]:
class BayesianOptimizer:
    def __init__(self, is_noisy=False):
        """
        Initializes the optimizer step config.
        """
        # Function 2 handles a noisy likelihood with a robust alpha boundary
        self.alpha = 1e-2 if is_noisy else 1e-6
        self.model = None
        self.X = None
        self.Y = None
        self.dim = None

    def load_and_fit(self, func_dir):
        """
        Loads multi-round cumulative histories and updates the Gaussian Process Regression model.
        """
        X = np.load(os.path.join(func_dir, "initial_inputs.npy"))
        Y = np.load(os.path.join(func_dir, "initial_outputs.npy"))
        self.X, self.Y, self.dim = X, Y, X.shape[1]
        
        kernel = ConstantKernel(1.0) * Matern(
            length_scale=np.ones(self.dim), 
            length_scale_bounds=(0.01, 1.0), 
            nu=2.5
        )
        
        # n_restarts_optimizer is raised to 35 for higher fitting precision
        self.model = GaussianProcessRegressor(
            kernel=kernel, 
            alpha=self.alpha, 
            normalize_y=True,
            n_restarts_optimizer=35
        )
        self.model.fit(self.X, self.Y)

    def expected_improvement(self, x, xi=0.01):
        """
        Computes Expected Improvement (EI) surface evaluations using the adaptive xi modifier.
        """
        x = x.reshape(-1, self.dim)
        mu, sigma = self.model.predict(x, return_std=True)
        mu_sample_opt = np.max(self.Y)

        with np.errstate(divide='warn'):
            imp = mu - mu_sample_opt - xi
            Z = imp / sigma
            ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei[sigma <= 0.0] = 0.0
        return ei

    def propose_next_point(self, xi_value):
        """
        Maximizes the complex EI surface utilizing 50 distinct L-BFGS-B random restarts.
        """
        best_x = None
        max_ei = -1
        
        # In Round 4, restarts are increased to 50 to conquer complex non-convex multimodal spaces
        for _ in range(50):
            x0 = np.random.uniform(0.0, 0.999999, self.dim)
            res = minimize(
                lambda x: -self.expected_improvement(x, xi=xi_value),
                x0,
                bounds=[(0.0, 0.999999)] * self.dim,
                method='L-BFGS-B'
            )
            if -res.fun > max_ei:
                max_ei = -res.fun
                best_x = res.x
        return best_x

## Loop Execution and Submission Generation (Functions 1-8)

In [ ]:
output_file = "submission_4_results.txt"

# Map out the dynamic jitter strategy values corresponding to current convergence trends
xi_map = {
    1: 0.100,  # Stuck at 10^-232, need massive exploration 
    2: 0.010,  # Steady progress 
    3: 0.100,  # Negative result, need to move 
    4: 0.100,  # Negative result, need to move 
    5: 0.001,  # EXPLOIT: Found huge peak (4440.48) 
    6: 0.100,  # Negative result, need to move 
    7: 0.001,  # EXPLOIT: Consistent growth 
    8: 0.001   # EXPLOIT: Found high value (9.88) 
}

print("Calculating Round 4 Queries (Climbing & Exploration Phase)...")
print("-" * 50)

notebook_results = {}

with open(output_file, "w") as f:
    for i in range(1, 8 + 1):
        func_dir = f"function_{i}"
        
        if not os.path.exists(func_dir):
            print(f"Warning: Directory '{func_dir}' not found. Skipping.")
            continue
            
        optimizer = BayesianOptimizer(is_noisy=(i == 2))
        optimizer.load_and_fit(func_dir)
        
        # Target point proposal with specific dynamic exploration metrics
        next_point = optimizer.propose_next_point(xi_value=xi_map[i])
        formatted_point = "-".join([f"{val:.6f}" for val in next_point])
        
        result_line = f"Function {i}: {formatted_point}"
        f.write(result_line + "\n")
        
        notebook_results[f"Function {i}"] = formatted_point
        print(f"{result_line} (used xi={xi_map[i]})")

print("-" * 50)
print(f"Submission 4 successfully generated and saved to: {output_file}")

### Summary Table of Proposed Queries

In [ ]:
print(f"| {'Target Objective':16} | {'Proposed Query Coordinates (Round 4)':55} |")
print(f"| {'-'*16} | {'-'*55} |")
for func, points in notebook_results.items():
    print(f"| {func:16} | {points:55} |")